# 15 Transports Views

Build the session from `nbplay.views` instead of the traitlets widgets. The views render the same elements, but each one hosts a pydantic config model in the kernel and a status model the browser owns, synced by [transports](https://github.com/1kbgz/transports) and rendered by [spaday](https://github.com/1kbgz/spaday). Config edits made in the browser are validated in the kernel before they land, and browser-side state (play, position, active pads) reaches Python without a traitlets round-trip.

Undo, named patterns, MIDI files and `save()`/`load()` work as on the traitlets `Session`; notebooks 11 and 13 show them.

In [ ]:
import math

import ipywidgets as widgets
from IPython.display import display

from nbplay import KeyboardRoute
from nbplay.views import PadsView, SamplerView, SequencerView, Session, SynthView


def decaying_tone(freq=440, seconds=0.22, sample_rate=44100, drive=0.9):
    frames = int(sample_rate * seconds)
    return [math.sin(2 * math.pi * freq * i / sample_rate) * drive * (1 - i / frames) ** 2 for i in range(frames)]


session = Session(bpm=120.0, time_signature=(4, 4))

lead = SynthView(oscillator_type="saw", frequency=440.0, amplitude=0.4)
lead_seq = SequencerView(length=16, num_voices=2)
for i, note in enumerate([72, 76, 79, 84, 83, 79, 76, 72, 71, 74, 78, 83, 81, 78, 74, 71]):
    lead_seq.set_step(i, note=note, velocity=88 + (i % 4) * 6, active=i % 2 == 0, voice=0)
for i, note in enumerate([60, 60, 64, 64, 67, 67, 64, 64, 59, 59, 62, 62, 66, 66, 62, 62]):
    lead_seq.set_step(i, note=note, velocity=70, active=i % 4 == 0, voice=1)

drums = SamplerView(attack=0.002, decay=0.06, sustain=0.2, release=0.08, max_voices=8)
drums.load_sample(decaying_tone(96, seconds=0.28, drive=1.0), sample_rate=44100, root_note=36, name="Kick-ish")
drums.configure_pads(pad_notes=[36, 38, 42, 46])
drum_seq = SequencerView(length=16)
for i in range(16):
    if i % 4 == 0:
        drum_seq.set_step(i, note=36, velocity=110)
    elif i % 4 == 2:
        drum_seq.set_step(i, note=38, velocity=96)
    else:
        drum_seq.set_step(i, note=42, velocity=60)

lead_track = session.add_track("Lead", lead_seq, lead)
drum_track = session.add_track("Drums", drum_seq, drums)

pads = PadsView(rows=1, cols=4, pad_notes=[36, 38, 42, 46], session_id=session.session_id)
pads.config.sampler_routing = [KeyboardRoute(drum_track.mixer_channel).to_dict()]

display(widgets.VBox([session.transport, lead_seq, lead, drum_seq, drums, pads, session.mixer]))

Press play on the transport, or call `session.play()` from the kernel. Both go through the shared browser clock, so the sequencers start together either way. The transport's status model mirrors the clock back to Python: `session.transport.status.is_playing` and `current_beat` update as the browser runs.

Every view keeps its configuration in `view.config`. Assigning a field validates the whole model and pushes the change to the browser; a change made in the browser (dragging the lead's frequency, toggling a step) arrives in the same object.

In [ ]:
session.play()
session.bpm = 126.0
lead.config.frequency = 330.0
print("bpm:", lead_seq.config.bpm, drum_seq.config.bpm)
print("lead frequency:", lead.config.frequency, "waveform bytes:", len(lead.config.waveform))
print("playing:", session.transport.status.is_playing)

## Launcher and timeline

The session owns a launcher and a timeline too. Launcher slots take a `SequencerView` (or step dicts) and launches are commands to the browser, quantized on the clock; timeline lanes were added by `add_track`, and MIDI clips land on them from the kernel.

In [ ]:
session.launcher.add_scene("A")
session.launcher.set_slot(lead_track.mixer_channel, 0, lead_seq, name="Lead A")
session.launcher.set_slot(drum_track.mixer_channel, 0, drum_seq, name="Drums A")
session.launcher.launch_scene(0)

session.timeline.add_midi_clip(
    "Riff",
    track_index=lead_track.mixer_channel,
    start=4.0,
    events=[{"beat": beat, "note": note, "velocity": 96, "duration": 0.5} for beat, note in enumerate([72, 76, 79, 84])],
)
display(widgets.VBox([session.launcher, session.timeline]))

## Theme and teardown

`session.theme` writes every view's `theme` field; the browser restyles in place. `session.close()` closes the views and their transports sessions.

In [ ]:
session.theme = "studio"
session.stop()
print("views:", len(session.views), "theme:", session.theme)